# Setup

Before running this notebook, complete the steps below.

**1. Install** — see the [README](../README.md) (`uv sync` or `pip install .`).

**2. Download the data** from Zenodo:
[doi.org/10.5281/zenodo.18798600](https://doi.org/10.5281/zenodo.18798600)
(Open access — click **Download all** on the record page.)

This dataset accompanies the paper:
> Leemans, Fako, Zaza, Schwaller & Buonsanti,
> "Ligand-Modulated Release of Copper Active Sites Extends Ethylene Production
> in CO2 Electroreduction", *JACS* (2026) —
> [doi.org/10.1021/jacs.5c22701](https://doi.org/10.1021/jacs.5c22701)

Extract all downloaded files into a single directory:
```bash
mkdir -p /path/to/dataset_jacs2026
# move / unzip the Zenodo files there
```

**3. Fill in the paths** in the Configuration cell below and run it.
No shell exports or kernel restarts needed.

In [ ]:
import os

# ── Paths — edit these to match your local setup ──────────────────────────────
os.environ.setdefault('JACS_DATA_DIR',   '/path/to/dataset_jacs2026')  # extracted Zenodo files
os.environ.setdefault('CFT_SCRATCH_DIR', '.')                           # output dir for figures / PLY files

## Init

In [1]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns

from glob import glob
import numpy as np
from ase.visualize import view
from pathlib import Path

from ase.io import  read, write

from cft import Manifold
from cft.mesh_utils import values_to_colors

# Set all fonts to Arial size 7/8
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 8,
    'axes.titlesize': 8,
    'axes.labelsize': 8,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 7
})

torch-sim-atomistic not installed, defaulting to sequential optimization


## Set dataset location

In [2]:
_jacs_data = os.environ.get('JACS_DATA_DIR')
if not _jacs_data:
    raise EnvironmentError("JACS_DATA_DIR is not set. Set it to your dataset_jacs2026 directory.")
DATA_DIR = Path(_jacs_data)
SCRATCH_DIR = Path(os.environ.get('CFT_SCRATCH_DIR', '.'))


OSError: JACS_DATA_DIR is not set. Set it to your dataset_jacs2026 directory.

## Read data

In [3]:
dir = DATA_DIR / 'Cu_smash' / '*_*' / 'relaxed_*.xyz'
files = glob(dir.as_posix())

files.sort()

ref_en_dict = {}

frames = []
for file in files:

    key = file.split('/relaxed_')[-1].split('.xyz')[0]
    atoms = read(file)
    en = atoms.get_potential_energy()
    
    ref_en_dict['grd_'+key] = en

    frames.append(atoms)

ref_en_dict

{'grd_top0': np.float64(-8012.22314453125),
 'grd_top1': np.float64(-8312.166015625),
 'grd_top2': np.float64(-7870.5888671875),
 'grd_top3': np.float64(-7720.6923828125),
 'grd_top4': np.float64(-8015.201171875),
 'grd_top12': np.float64(-7869.8955078125),
 'grd_top13': np.float64(-7720.63720703125),
 'grd_top14': np.float64(-8014.357421875),
 'grd_top15': np.float64(-7720.73583984375),
 'grd_top16': np.float64(-7722.091796875),
 'grd_top10': np.float64(-8164.275390625),
 'grd_top6': np.float64(-8011.2099609375),
 'grd_top7': np.float64(-7723.0087890625),
 'grd_top8': np.float64(-7575.80810546875),
 'grd_top9': np.float64(-8309.9482421875)}

In [ ]:
naked_atoms = atoms[[atom.index for atom in atoms if atom.symbol =='Cu']]

m = Manifold(
    naked_atoms,
    mode='particle',
    precision =1.5,
    touch_sphere_size = 2.5
)

## Write PLY files

In [ ]:
i=9
k='delta_e_O_0'
k='rel_e_ClC#O+_0'
k='ref_e_ClC#O+_0'
range_cut = 1.2


for i, _ in enumerate(trj): #from save ovito
    for k in trj[0].arrays.keys():

        # if '_H_' not in k:
        #     continue

        # view(frames[i])
        atoms = frames[i]
        write(str(SCRATCH_DIR / f'atoms_{i}.xyz'), atoms)

        vals = trj[i].arrays[k]
        if 'delta_' in k:
            vals[vals>.4] = 0.4
            vals[vals<-.4] = -.4
            palette_nam='bwr'
        elif 'rel_' in k:
            vals[vals>range_cut] = range_cut #vals[vals>1] = .5
            vals[vals<0] = 0.
            palette_nam='plasma'
        elif 'ref_' in k:
            vals[vals>range_cut] = range_cut
            vals[vals<0] = 0.
            palette_nam='plasma'
        else:
            continue
        
        # if 'delta_e_H_' in k:
        # # if 'ClC#O+' in k:
        
        #     print(k)
        #     sns.kdeplot(vals.flatten(), label=k, legend=True)
        #     colors = [values_to_colors(v, vals, palette_nam=palette_nam) for v in vals]
        #     m.save_ply(filename=str(SCRATCH_DIR / f'ply_{k}_{i}.ply'), vertex_colors=colors)

NameError: name 'trj' is not defined

In [ ]:
fig, axs = plt.subplots(5,3)

axs = axs.flatten()

for i, a in enumerate(trj):

    ax= axs[i]
    
    xs= list(a.arrays['rel_e_H_0'])
    ys= list(a.arrays['rel_e_ClC#O+_0'])

    sns.kdeplot(x=xs, y=ys, ax=ax)
    # ax.hexbin(xs, ys, gridsize=50, cmap="viridis")
    # sns.jointplot(x=xs, y=ys, kind="hist", bins=60, ax=ax)
    # plt.colorbar()


    # xs = list(a.arrays['ref_e_H_0'])
    # ys = list(a.arrays['ref_e_ClC#O+_0'])
    # sns.kdeplot(x=xs, y=ys, ax=ax, color='k')

    # ax.set_xlim(0,.8)
    # ax.set_ylim(-0.5,5)

In [ ]:
# e_H_0
# e_C_0
# e_O_0
# e_ClC#O+_0
# e_ClP_0
# delta_e_H_0
# ref_e_H_0
# rel_e_H_0
# delta_e_C_0
# ref_e_C_0
# rel_e_C_0
# delta_e_O_0
# ref_e_O_0
# rel_e_O_0
# delta_e_ClC#O+_0
# ref_e_ClC#O+_0
# rel_e_ClC#O+_0
# delta_e_ClP_0
# ref_e_ClP_0
# rel_e_ClP_0
i=9

k = 'ref_e_ClC#O+_0'
# k = 'delta_e_H_0'
vals = trj[i].arrays[k]
# _, curvature_magnitude = curvature_deformed_cube(m.grid, m.faces, m.normals)
# plt.scatter(_, vals, alpha=0.2)

In [ ]:
from cft.mesh_utils import slice_atoms_near_point

In [ ]:
grd = trj[0]
def get_if_interface(naked_atoms, grid, d=6):
    interface = []
    for atom in grid:
        slice_atoms = slice_atoms_near_point(naked_atoms, atom.position, d=6)
        interface.append(len(set(slice_atoms.arrays['fragments'])))
    return np.array(interface)


In [ ]:
naked_atoms.arrays.keys()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

valsx_ref = trj[i].arrays['ref_e_H_0']
valsy_ref = trj[i].arrays['ref_e_ClC#O+_0']
# valsx_rel = trj[i].arrays['rel_e_ClC#O+_0']
# valsy_rel = trj[i].arrays['rel_e_H_0']

valsx = trj[i].arrays['rel_e_H_0']
valsy = trj[i].arrays['rel_e_ClC#O+_0']
_=valsy.copy()
valsx_rel = valsx[_<1.]
valsy_rel = valsy[_<1.]


# Two distinct plasma colors
# cmap = plt.cm.plasma
# color_ref = cmap(0.2)   # darker violet
# color_rel = cmap(0.65)   # bright yellow

val1 = .1
val2 = 0.2
color_ref = (val1, val1, val1, 1.)   
color_rel = (val2, val2, val2, .3)   


fig, axes = plt.subplots(2, 2, figsize=(100/25.4*1.2, 100/25.4), dpi=600)

# ax = axes[0,1]
ax = axes[0,0]

cset = sns.kdeplot(
    x=valsx_rel, y=valsy_rel, fill=False, color=color_rel,ax=ax
    )
sns.kdeplot(
    x=valsx_ref, y=valsy_ref, fill=False, color=color_ref,ax=ax # alpha=1.
    )
for line in cset.collections:
    line.set_linewidth(0.9)

ax.set_xlabel("E$_{ads. relative} (*H)$ / eV")
ax.set_ylabel("E$_{ads. relative} (*CO)$ / eV")

ax.set_xlim(0, .9)
ax.set_ylim(-.3, 1.1)


##################### INTRFACE

ax = axes[1,0]

interface_array = get_if_interface(naked_atoms, trj[i], d=6)

valsx_interface = trj[i].arrays['ref_e_H_0'][interface_array==2]
valsy_interface = trj[i].arrays['ref_e_ClC#O+_0'][interface_array==2]

valsx_normal = trj[i].arrays['ref_e_H_0'][interface_array==1]
valsy_normal = trj[i].arrays['ref_e_ClC#O+_0'][interface_array==1]


cset = sns.kdeplot(x=valsx_interface, y=valsy_interface, fill=False,color=color_rel,ax=ax)
sns.kdeplot(x=valsx_normal, y=valsy_normal, fill=False, ax=ax, color=color_ref)
# sns.kdeplot(x=valsx_ref, y=valsy_ref, fill=False,ax=ax, color=color_ref)
for line in cset.collections:
    line.set_linewidth(0.9)


# sns.kdeplot(x=valsx_rel, y=valsy_rel, fill=False,colors=colors,alpha=0.6,ax=ax)

ax.set_xlabel("E$_{ads. relative} (*H)$ / eV")
ax.set_ylabel("E$_{ads. relative} (*CO)$ / eV")

ax.set_ylim(-.3, 1.1)
ax.set_xlim(0, .9)

###################

ax = axes[0,1]
# ax = axes[1,1]

vals = []
es = []
for e in np.arange(-.1, 1, 0.01):
    vals+=[np.sum((trj[i].arrays['ref_e_H_0'] < e) * m.grid_area)]
    es+=[e]
ax.plot(es, vals, c=color_ref, linestyle='--'); print(max(vals))

vals = []
es = []
for e in np.arange(-.1, 1, 0.01):
    vals+=[np.sum((trj[i].arrays['ref_e_ClC#O+_0'] < e) * m.grid_area)]
    es+=[e]

ax.plot(es, vals, c=color_ref)#'r')
ax.axhline(max(vals), linestyle='--', color='k', linewidth=.5, alpha=.6); print(max(vals))

ax.set_xlabel("Active Area (no ligand) / Å²")
ax.set_ylabel("E$_{ads. relative}$ / eV")
ax.set_xlim(-100, 4000)
ax.set_ylim(-.1, 1)

###################


vals = []
es = []
for e in np.arange(-.1, 1, 0.01):
    vals+=[np.sum((trj[i].arrays['rel_e_H_0'] < e) * m.grid_area)]
    es+=[e]
ax.plot(es, vals, c=color_rel, linestyle='--')#'b'); 
ax.axhline(max(vals), linestyle='--', color='k', linewidth=.5); print(max(vals))

vals = []
es = []
for e in np.arange(-.1, 1, 0.01):
    vals+=[np.sum((trj[i].arrays['rel_e_ClC#O+_0'] < e) * m.grid_area)]
    es+=[e]

ax.plot(es, vals, c=color_rel)
ax.axhline(max(vals), linestyle='--', color='k', linewidth=.5); print(max(vals))

ax.set_ylabel("Active Area / Å²")
# ax.set_ylabel("Active Area (with ligand) / Å²")
ax.set_xlabel("E$_{ads. relative}$ / eV")
ax.set_ylim(-100, 4000)
ax.set_xlim(-.1, 1)


plt.tight_layout()
plt.show()

In [ ]:
# xxx = trj[9].copy()
# xxx.symbols = get_if_interface(naked_atoms, xxx, d=6)

# view(xxx)

In [ ]:
print(max(vals))

In [ ]:
vals = []
es = []
for e in np.arange(0, 1, 0.01):
    vals+=[np.sum((trj[i].arrays['rel_e_H_0'] < e) * m.grid_area)]
    es+=[e]
plt.plot(vals, es, c='b')

vals = []
es = []
for e in np.arange(0, 1, 0.01):
    vals+=[np.sum((trj[i].arrays['rel_e_ClC#O+_0'] < e) * m.grid_area)]
    es+=[e]

plt.plot(vals, es, c='r')

In [ ]:
valsx = trj[i].arrays['rel_e_ClC#O+_0']
_=valsx.copy()
valsy = trj[i].arrays['rel_e_H_0']
valsx = valsx[_<1.]
valsy = valsy[_<1.]

plt.scatter(valsx, valsy, alpha=0.05)

In [ ]:
view(grid_atoms + trj[0])

In [ ]:
view(atoms)

In [ ]:
# view(read(DATA_DIR / 'Cu_smash/naked_particle.xyz'))

## si

In [ ]:
df = pd.read_csv(DATA_DIR / 'Cu_smash/MD_statistics/md_data.csv')

In [ ]:
df['system'] = df['name'] + '-' + df['n_atoms'].astype(str)
df['E total (per atom)'] = df['E_tot'] /  df['n_atoms']
df['simulation %'] = df.frame / df.groupby('file')['frame'].transform('max')
df['time'] = df.frame*500*1e-6


In [ ]:
colors = {
    'sphere-262':      "#4C72B0",  # blue
    'tetrahedron-214': "#DD8452",  # orange
    'rhombic-220':     "#55A868",  # green
    'octahedron-340':  "#C44E52",  # red
    'cube-388':        "#8172B2",  # purple
}

# sample evenly across the plasma colormap
cmap = plt.get_cmap("plasma")
colors = {sys: cmap(i / (len(systems)+6)) for i, sys in enumerate(systems)}
colors

In [ ]:
dict_size = {
    'sphere-184': 200,
    'sphere-262': 200,
    'sphere-596': 200,
    'sphere-932': 200,
    'sphere-1808': 1000,
    'sphere-3660': 1000,
    }

In [ ]:
systems = ['sphere-184',
# 'sphere-262',
'sphere-596', 'sphere-932', 'sphere-1808', 'sphere-3660']
# systems = ['sphere-262', 'tetrahedron-214', 'rhombic-220', 'octahedron-340', 'cube-388']

fig, axs = plt.subplots(len(systems),3, figsize=[180/25.4, len(systems)*35/25.4], dpi=300) #, sharex='col'

for j, system in enumerate(systems):
    # if j>1:
    #     break
    pdf = df[(df.system == system) & (df.frame > 3) & (df.frame < dict_size[system])]

    run_count = len(pdf.file.unique())
    
    sns.lineplot(pdf, x='time', y='interface_interactions', ax=axs[j,0], color=colors[system])
    axs[j,0].set_xlabel('MD step / ns'); axs[j,0].set_ylabel('interface / -')
    for frc in [.0, .1, .5, 1.]:
        frame = int(dict_size[system] * frc) * 500*1e-6
        axs[j,0].axvline(x=frame, color='k', linestyle='--', linewidth=.6)

    sns.lineplot(pdf, x='time', y='E total (per atom)', ax=axs[j,1], color=colors[system])
    axs[j,1].set_xlabel('MD step / ns'); axs[j, 1].set_ylabel(r"$E_{\mathrm{total}}$ (per atom) / eV")
    
    sns.lineplot(pdf, x='time', y='T', ax=axs[j,2], color=colors[system])
    axs[j,2].set_xlabel('MD step / ns'); axs[j, 2].set_ylabel("T / K")
    

    axs[j,0].text(
        0.92, 0.05,           # x, y in axis coordinates (lower-right)
        f"{system}\n{run_count} runs",
        transform=axs[j,0].transAxes,
        ha='right', va='bottom',
        fontsize=7,
        fontweight='bold',
        color=colors[system]
    )

fig.set_layout_engine(layout='tight')

In [ ]:
for frc in [.0, .1, .5, 1.]:
    frame = int(dict_size[system] * frc)
    print(frame*500 * 1e-6)

In [ ]:
trj_dict = {}

for j, system in enumerate(systems):
    pdf = df[(df.system == system) & (df.frame > 3) & (df.frame < dict_size[system])]
    file = pdf.file.values[0]
    trj_dict[system] = read(file, index=':')

In [ ]:
systems

In [ ]:
from ase import Atoms
render_atoms = Atoms()


for j, system in enumerate(systems):
    
    trj =  trj_dict[system]

    slide = 0
        
    if j > 0:
        slide = min(render_atoms.positions[:,2])-25
    
    
    for i, frc in enumerate([.0, .1, .5, 1.]):
        frame = int(dict_size[system] * frc)

        atoms = trj[frame].copy()
        atoms.positions += np.array([75,0,0])*i
        
        atoms.positions += [0,0,slide] 
        # atoms.positions += np.array([0,0,-35])*j
        render_atoms += atoms

view(render_atoms)
        
        

In [ ]:
for atom in render_atoms:
    if render_atoms.arrays['fragments'][atom.index] == -1:
        atom.symbol = 'Zn'

write(str(SCRATCH_DIR / 'render_atoms.xyz'), render_atoms)

### ligands

In [10]:
import os

In [ ]:
glob_str = DATA_DIR / 'Cu_smash/*/relaxed*top*'
glob_str = glob_str.as_posix()

files = glob(glob_str)
files.sort()
files

['/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/0_5/relaxed_top0.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/0_5/relaxed_top1.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/0_5/relaxed_top2.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/0_5/relaxed_top3.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/0_5/relaxed_top4.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/12_17/relaxed_top12.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/12_17/relaxed_top13.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/12_17/relaxed_top14.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/12_17/relaxed_top15.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/12_17/relaxed_top16.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/6_11/relaxed_top10.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/dataset_jacs2026/Cu_smash/6_11/relaxed_top6.xyz',
 '/mnt/c/Users/ef/Desktop/tmp/data

In [17]:
from ase import Atoms
render_atoms = Atoms()

base_dir = str(DATA_DIR / 'Cu_smash') + '/'


for j, folder in enumerate(['0_5', '6_11', '12_17']):
    
    files = glob(os.path.join(base_dir, folder, 'relaxed*xyz'))
    files.sort()

    trj = [read(file) for file in files]
    
    
    for i, a in enumerate(trj):
        
        atoms = a.copy()
        atoms.positions-=atoms.get_center_of_mass()
        atoms.positions += np.array([0,0,35])*i
        
        atoms.positions += np.array([55,0,0]) *j
        # atoms.positions += np.array([0,0,-35])*j
        render_atoms += atoms

view(render_atoms)
        
        

<Popen: returncode: None args: ['/home/ef/Code/CFT/.venv/bin/python', '-m', ...>

In [ ]:
view(trj)

<Popen: returncode: None args: ['/home/ef/Code/CFT/.venv/bin/python', '-m', ...>